# 07b — Grounding, permissions, and vector search — Practice

Read the lesson and run the example. Implement each TODO, then run its checks. A NotImplementedError is expected until you write your answer. Try for 20–30 minutes before opening a hint; open the solution only after recording an attempt. Copy this notebook into work/ to keep your own version.

**Prerequisites:** earlier lessons in the course order. **Environment:** Install requirements/core.txt.

[Stage guide](../stages/07.md) · [Course map](../PLAN.md) · [Project](../../projects/stage07/README.md)

**Notebook time:** 2–4 hours for an initial pass; project, reading, retrieval practice, and independent transfer use the rest of the stage budget.

Learning objectives:

- Filter access before ranking
- Return traceable citations and abstentions
- Use actual embeddings and pgvector in the project


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets" / "financials.csv").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the learning repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

def expect_error(error_type, operation):
    try:
        operation()
    except error_type:
        return
    raise AssertionError(f"Expected {error_type.__name__}")

print("Course root:", ROOT)


## Lesson

RAG combines retrieved evidence with generation. It cannot guarantee truth: wrong sources, stale versions, weak retrieval, or unsupported synthesis can each cause failure. Return an explicit abstention when the evidence does not support the question. Separate retrieval recall from citation support and answer correctness.

Permissions must constrain the candidate set before evidence reaches a model. Filtering only the displayed citations is too late. Caches, rerankers, logging, and backups also need tenant boundaries. A vector database stores embeddings and metadata; similarity is meaningful only when the embedding model, dimension, preprocessing, and distance convention agree.

Approximate indexes trade recall for speed. Test recall after metadata filtering and inspect query plans. The notebook uses TF-IDF vectors to teach the interface offline; the project includes learned sentence embeddings, PostgreSQL/pgvector, and a model adapter. These are distinct experiments and must be reported as such.


## Worked example

Predict the output before running. Change one input and explain the result.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
texts = ["revenue grew", "password reset"]
x = TfidfVectorizer().fit_transform(texts)
print(x.shape)


## Exercise 1: Authorized candidates

Return public documents or documents with tenant equal to the authenticated tenant. Do not infer identity from the query text.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def authorized(documents, tenant):
    raise NotImplementedError


In [ ]:
docs = [{"id": "a", "tenant": "a"}, {"id": "b", "tenant": "b"}, {"id": "p", "tenant": "b", "public": True}]
assert [d["id"] for d in authorized(docs, "a")] == ["a", "p"]
print("Exercise 1: checks passed")


<details><summary>Hint — open after trying</summary>

Authorization is a server-side predicate.

</details>


## Exercise 2: Cosine vector ranking

Using TF-IDF, rank authorized nonzero-scoring documents. Return (id,score) pairs, sorted descending then id. Empty input returns [].

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def vector_search(query, documents, tenant, k=3):
    raise NotImplementedError


In [ ]:
docs = [{"id": "a", "tenant": "A", "text": "annual revenue"}, {"id": "b", "tenant": "B", "text": "secret revenue"}]
assert vector_search("revenue", docs, "A")[0][0] == "a"
assert vector_search("missing", docs, "A") == []
print("Exercise 2: checks passed")


<details><summary>Hint — open after trying</summary>

TF-IDF normalizes rows by default, so dot products are cosine scores.

</details>


## Exercise 3: Evidence contract

Return status=unsupported with empty citations when no hits; else status=evidence with exact quotes and IDs. Do not synthesize a claim.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def evidence_response(hits, by_id):
    raise NotImplementedError


In [ ]:
assert evidence_response([], {}) == {"status": "unsupported", "citations": []}
assert evidence_response([("a", .5)], {"a": {"text": "Revenue 10"}})["citations"][0]["quote"] == "Revenue 10"
print("Exercise 3: checks passed")


<details><summary>Hint — open after trying</summary>

Preserve the exact evidence before adding a generator.

</details>


## Independent transfer

Run the pgvector lab, ingest Markdown plus one PDF, and answer through Ollama. Evaluate access isolation, paraphrases, exact numbers, unsupported questions, and malicious retrieved instructions.

Record your implementation, errors, measurements, and explanation in work/. See the project’s ANSWERS.md after attempting the brief.


## Explain without code

1. Can a citation be valid but misleading?

2. Why filter before model access?


## Reading and review

- [Primary reference 1](https://github.com/pgvector/pgvector)
- [Primary reference 2](https://docs.ollama.com/api/chat)

After 2, 7, and 30 days: explain the concept from memory, solve a changed-input version, and log any gaps in progress.md.
